In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))

import config
import pandas as pd
df = pd.read_csv(config.DB_LOCATION)

C:\Users\bnpar\AppData\Local\Temp\ipykernel_13772\2436533794.py:10: DtypeWarning: Columns (33,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(config.DB_LOCATION)


In [2]:
#reduced dataframe
red_df = df[['Name', 'Sex', 'Division', 'BodyweightKg', 'WeightClassKg','Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg', 'TotalKg', 'Date', 'Sanctioned', 'Equipment', 'Event']]

### Cleaning dataset



In [3]:
def get_mixed_cols(dataframe):
    all_types_count = {} #nested dict w column heading as outer key. Inner key-value pairs are data types/how many entries with that datatyoe
    for col in dataframe.columns:
        col_types = dataframe[col].map(type)   #applies type(x) to every x in the column so you get series ['string', 'string', 'float'....]
        all_types_count[col] = col_types.value_counts().to_dict()
    mixed_type_cols = []
    for col in all_types_count:
        no_types = len(all_types_count[col].values())
        #we are only appending columns with mixed data types
        if no_types > 1:
            mixed_type_cols.append(col)
    return mixed_type_cols

mixed_cols = get_mixed_cols(red_df)
print(mixed_cols)

['Division', 'WeightClassKg']


In [4]:
#checking if all the floats are because of NaN
for col in mixed_cols:
    mask = red_df[col].map(type) == float
    float_col = red_df.loc[mask, col]
    float_col = float_col.dropna()
    print(float_col)

Series([], Name: Division, dtype: object)
Series([], Name: WeightClassKg, dtype: object)


- All float datatypes are because of NaN entries in WeightClassKg or Division. Want to drop any entries where we dont know what the weight class is.
- Are okay with not knowing the Division i.e. whether the lifter is a Junior or Open lifter for now, as a lifter of any age can set an Open WR.
- Chosen to drop any unsanctioned meets.
- Flitered for Raw powerlifting only (specified in 'Equipment' column).
- Turned dates into datetime objects.
- Filtering for full power events (e.g. excluding bench only comps)

In [5]:
red_df = red_df[red_df['Equipment'] == 'Raw'].copy()
red_df['Date'] = pd.to_datetime(red_df['Date'] , format = '%Y-%m-%d')
red_df['Year'] = red_df['Date'].dt.year
red_df = red_df.dropna(axis = 'rows', subset = ['WeightClassKg'])
unsanctioned = red_df[red_df['Sanctioned'] == 'No'].index.to_list()
red_df = red_df.drop(labels = unsanctioned, axis = 'index')
full_power = red_df[red_df['Event'] == 'SBD'].copy()
#bench_only = red_df[red_df['Event']== 'B'].copy()


Next we flag entries in the dataset that exceed the current world official world record. Note that the total being more than the official WR does not necessarily mean the entry is incorrect. Official WRs can only be set at international meets and lifters may have exceeded this at national or local events, without setting a new official WR.

Also note that the 53kg and 43kg in the men's and women's categories respectively are only in the Junior age division.


In [6]:
m_off_wr_totals = {'53': 559, '59': 669.5, '66': 770, '74': 843, '83': 890, '93': 918, '105': 975.5, '120': 978.5, '120+': 1152.5}
f_off_wr_totals = {'43': 349.5, '47': 435, '52': 482.5, '57': 520, '63': 565, '69': 628, '76': 620, '84': 647, '84+': 737.5}

exceeds_wr = pd.DataFrame(columns = red_df.columns.to_list())

for weight_class in m_off_wr_totals:
    weight_class_entries = full_power[(full_power['WeightClassKg'] == weight_class) & (full_power['Sex'] == 'M')].copy()
    exceeds_class_wr = weight_class_entries[weight_class_entries['TotalKg']>m_off_wr_totals[weight_class] ]
    exceeds_wr = pd.concat([exceeds_wr, exceeds_class_wr])

for weight_class in f_off_wr_totals:
    weight_class_entries = full_power[(full_power['WeightClassKg'] == weight_class) & (full_power['Sex'] == 'F')].copy()
    exceeds_class_wr = weight_class_entries[weight_class_entries['TotalKg']>f_off_wr_totals[weight_class]]
    exceeds_wr = pd.concat([exceeds_wr, exceeds_class_wr])
    
exceeds_wr

C:\Users\bnpar\AppData\Local\Temp\ipykernel_13772\1112979477.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  exceeds_wr = pd.concat([exceeds_wr, exceeds_class_wr])


,Name,Sex,Division,BodyweightKg,WeightClassKg,Best3SquatKg,Best3BenchKg,Best3DeadliftKg,TotalKg,Date,Sanctioned,Equipment,Event,Year
891163,Sergey Fedosienko,M,Open,58.50,59,226.5,170.5,273.0,670.0,2015-12-24,Yes,Raw,SBD,2015
907319,Sergey Fedosienko,M,Open,59.00,59,228.0,167.5,275.0,670.5,2020-03-18,Yes,Raw,SBD,2020
1242982,Joseph Borenstein,M,MR-O,82.91,83,305.0,218.0,377.0,900.0,2025-04-03,Yes,Raw,SBD,2025
1237528,William Ball #2,M,MR-Jr,92.15,93,322.5,222.5,375.0,920.0,2025-09-27,Yes,Raw,SBD,2025
1243024,Jonathan Cayco,M,MR-O,92.85,93,310.0,252.0,357.5,919.5,2025-04-03,Yes,Raw,SBD,2025
1243068,Anthony McNaughton,M,MR-O,103.74,105,365.0,245.0,369.0,979.0,2025-04-03,Yes,Raw,SBD,2025
1241212,Bobb Matthews,M,MR-O,110.75,120,367.5,247.5,395.5,1010.5,2025-11-22,Yes,Raw,SBD,2025
1243132,Rondel Hunte,M,MR-G,119.58,120,375.0,260.0,375.0,1010.0,2025-04-03,Yes,Raw,SBD,2025
1241213,Jesus Olivares,M,MR-O,182.40,120+,478.5,265.0,410.0,1153.5,2025-11-22,Yes,Raw,SBD,2025
937711,Joy Nnamani,F,FR-O,56.70,57,185.0,100.0,235.5,520.5,2025-12-05,Yes,Raw,SBD,2025


All entries that exceed the current world record are credible and correct against the results from those meets.

### Dealing with weight class changes

Next we look for weight classes in the df which are not the weight classes that the IPF and its affiliates currrently use. 


In [9]:
valid_f_class_mask = (full_power['WeightClassKg'].isin(f_off_wr_totals.keys())) & (full_power['Sex'] == 'F')
valid_m_class_mask = (full_power['WeightClassKg']).isin(m_off_wr_totals.keys()) & (full_power['Sex'] == 'M')

invalid_class_entries = full_power[(~valid_f_class_mask) & (~valid_m_class_mask) ].copy()
f_invalid_class_entries = full_power[~valid_f_class_mask].copy()
m_invalid_class_entries = full_power[~valid_m_class_mask].copy()

f'{(len(invalid_class_entries)/len(full_power))*100 :.2f}% of the dataset is entries that do not match the IPF\'s current weight classes'

"7.08% of the dataset is entries that do not match the IPF's current weight classes"

In [25]:
import numpy as np
f_invalid_class = invalid_class_entries[invalid_class_entries['Sex'] == 'F']['WeightClassKg'].unique()
#dictionary with F weight class as key, values are years that the class had participation
f_invalid_class_years = {}

m_invalid_class = invalid_class_entries[invalid_class_entries['Sex'] == 'M']['WeightClassKg'].unique()
#dictionary with M weight class as key, values are years that the class had participation
m_invalid_class_years = {}

for weight_class in f_invalid_class:
    class_df = f_invalid_class_entries[f_invalid_class_entries['WeightClassKg'] == weight_class]
    f_invalid_class_years.update({weight_class : class_df['Year'].unique()})

for weight_class in m_invalid_class:
    class_df = m_invalid_class_entries[m_invalid_class_entries['WeightClassKg'] == weight_class]
    m_invalid_class_years.update({weight_class : class_df['Year'].unique()})

class_count = full_power.groupby(['WeightClassKg', 'Sex']).size().reset_index(name = 'NumberOfEntries')
annual_participation = full_power.groupby(['Year', 'WeightClassKg', 'Sex']).size().reset_index(name = 'AnnualParticipation')

#dealing with F weight classes
f_percentages = {}
for weight_class in f_invalid_class:
    f_percentages.update({weight_class : {}})
    for year in f_invalid_class_years[weight_class]:
        total_ann_participation = np.sum(
            annual_participation[(annual_participation['Sex'] == 'F') & (annual_participation['Year'] == year)]['AnnualParticipation']
        )
        class_ann_participation = np.sum(
            annual_participation[(annual_participation['Sex'] == 'F') & (annual_participation['Year'] == year) & (annual_participation['WeightClassKg'] == weight_class)]['AnnualParticipation']
        )
        print(total_ann_participation, class_ann_participation, year, weight_class)
        perc = (class_ann_participation/total_ann_participation) *100
        f_percentages[weight_class].update({year: perc})
        

#dealing with M weight classes
    
    

1293 213 2012 72
19584 4373 2018 72
21363 4858 2019 72
5365 901 2014 72
13254 3086 2016 72
17095 4079 2017 72
2824 501 2013 72
9236 2092 2015 72
7296 1668 2020 72
819 122 2011 72
313 1 2010 72
12178 112 2021 72
20471 1 2024 72
0 0 1973 56
1 0 1983 56
0 0 1979 56
0 0 1975 56
0 0 1978 56
0 0 1977 56
0 0 1980 56
1 1 2000 56
313 35 2010 56
18 5 2004 56
197 27 2009 56
12 4 2002 56
40 11 2005 56
9 3 2003 56
19584 16 2018 56
5365 111 2014 56
2824 68 2013 56
17095 19 2017 56
1293 40 2012 56
13254 5 2016 56
123 13 2008 56
0 0 1972 56
0 0 1971 56
0 0 1966 56
53 8 2007 56
0 0 1967 56
0 0 1969 56
0 0 1974 56
0 0 1970 56
38 1 2006 56
819 27 2011 56
0 0 1990 56
0 0 1976 56
2 0 1993 56
0 0 1974 90
9 1 2003 90
197 8 2009 90
313 8 2010 90
12 0 2002 90
40 0 2005 90
18 1 2004 90
19584 17 2018 90
123 8 2008 90
5365 62 2014 90
2824 32 2013 90
17095 17 2017 90
1293 28 2012 90
13254 2 2016 90
12178 0 2021 90
38 4 2006 90
53 3 2007 90
819 16 2011 90
1 0 1981 90
0 0 1980 90
1 0 1983 90
0 0 1984 90
9 1 2003 75


C:\Users\bnpar\AppData\Local\Temp\ipykernel_13772\2381524175.py:33: RuntimeWarning: invalid value encountered in scalar divide
  perc = (class_ann_participation/total_ann_participation) *100


123 28 2008 60
53 12 2007 60
819 34 2011 60
38 6 2006 60
0 0 1977 60
2 0 1993 60
0 0 1974 60
0 0 1997 67.5
197 43 2009 67.5
313 59 2010 67.5
12 1 2002 67.5
40 4 2005 67.5
9 1 2003 67.5
18 5 2004 67.5
19584 34 2018 67.5
5365 268 2014 67.5
2824 141 2013 67.5
17095 34 2017 67.5
1293 83 2012 67.5
13254 10 2016 67.5
123 26 2008 67.5
53 10 2007 67.5
38 7 2006 67.5
819 63 2011 67.5
0 0 1984 67.5
0 0 1996 67.5
2 0 1993 67.5
0 0 1974 67.5
21363 31 2019 72+
17095 11 2017 72+
19584 24 2018 72+
13254 1 2016 72+
7296 3 2020 72+
0 0 1973 82.5
9 0 2003 82.5
197 17 2009 82.5
313 31 2010 82.5
12 0 2002 82.5
40 4 2005 82.5
18 0 2004 82.5
19584 27 2018 82.5
123 8 2008 82.5
5365 130 2014 82.5
2824 62 2013 82.5
17095 26 2017 82.5
1293 16 2012 82.5
13254 6 2016 82.5
12178 0 2021 82.5
38 6 2006 82.5
53 2 2007 82.5
819 12 2011 82.5
0 0 1984 82.5
1 0 1982 82.5
1 0 1983 82.5
1 0 1981 82.5
2 1 1993 82.5
0 0 1974 82.5
17975 5 2023 37
20471 2 2024 37
13254 5 2016 30
17095 10 2017 30
7296 2 2020 30
21363 11 2019 30

In [18]:
annual_participation['AnnualParticipation'].dtype


dtype('int64')

In [ ]:
full_power[full_power['Sex']=='Mx']
#only 5 entries with Sex as 'Mx', split across 3 weight classes. can't draw inferences about future WRs from this

For each invalid weight class we flag if entries in that weight class make up less than 1% of total entries (per sex). 8 weight classes (or 9 in Junior age division) in the IPF. So less than 1% of entries should signal mistake.

Given that only 7.08% of the dataset are entries which do not match the IPF's current weight classes, these will be excluded from the dataset. Need to understand the impact of this so will determine when the weight class changes took place. There is no official documentation that describes the weight class changes. So need to look at the dataset to determine when these took place. 

We want the number of entries in each "invalid" class to determine whether the class was entered by mistake, or whether it is a historical weight class in the IPF. Will a

In [ ]:
#getting unique combinations of name, class and year
# unique_name_class_year = invalid_class_entries[['Name', 'WeightClassKg', 'Year', 'Sex']].drop_duplicates()
unique_name_class_year = invalid_class_entries[['Name', 'WeightClassKg', 'Year', 'Sex']]

invalid_class_counts = unique_name_class_year.groupby(['WeightClassKg', 'Year']).size().reset_index(name = 'NumberOfLifters').sort_values('Year')
all_class_counts = full_power[['Name', 'WeightClassKg', 'Year', 'Sex']]
all_class_counts = full_power.groupby(['WeightClassKg', 'Year']).size().reset_index(name = 'NumberOfLifters').sort_values('Year')

for w_class in invalid_class:
    years = (all_class_counts[all_class_counts['WeightClassKg'] == w_class]['Year']).unique()
    

In [ ]:
# want number of lifters competing in 'invalid' weight classes every year 
no_lifters_per_year = pd.DataFrame(columns = ['WeightClassKg', 'Year', 'NumberOfLifters'])

In [ ]:
# classes_per_year = pd.DataFrame(columns = ['Year', 'Classes'])



In [ ]:
mask = (red_df['TotalKg'].between(0,1153)).apply(lambda x: not x)
print(f'length is {len(red_df)}')
outliers = red_df[mask]
print(f'no outliers is {len(outliers)}')
no_bomb_out = outliers.dropna()
no_bomb_out.head(10)

In [ ]:
# copy = red_df.copy()
# copy['Year'] =  copy['Date'].dt.year
# copy['Year'].value_counts()